In [2]:
import pandas as pd
import numpy as np
from urllib.parse import urlparse
import re
import warnings
warnings.filterwarnings('ignore')
!pip install lightgbm
# ML Libraries
!pip install imblearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                           confusion_matrix, classification_report, roc_auc_score,
                           precision_recall_curve)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import joblib
from datetime import datetime
import matplotlib.pyplot as plt

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/3 [sklearn-compat]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- ------------------------

In [3]:
import seaborn as sns

print("="*80)
print("🚀 ENTERPRISE MALICIOUS URL DETECTOR - TRAINING ON YOUR DATA")
print("="*80)

# ============================================================================
# STEP 1: LOAD AND ANALYZE YOUR DATA
# ============================================================================

print("\n📂 STEP 1: Loading balanced_urls.csv...")
df = pd.read_csv('balanced_urls.csv')

print(f"\n✅ Dataset loaded successfully!")
print(f"   Total samples: {len(df)}")
print(f"   Columns: {df.columns.tolist()}")

# Display first few rows
print("\n📊 First 5 rows of your data:")
print(df.head())


🚀 ENTERPRISE MALICIOUS URL DETECTOR - TRAINING ON YOUR DATA

📂 STEP 1: Loading balanced_urls.csv...

✅ Dataset loaded successfully!
   Total samples: 632508
   Columns: ['url', 'label', 'result']

📊 First 5 rows of your data:
                         url   label  result
0     https://www.google.com  benign       0
1    https://www.youtube.com  benign       0
2   https://www.facebook.com  benign       0
3      https://www.baidu.com  benign       0
4  https://www.wikipedia.org  benign       0


In [4]:
print("\n📋 Column Information:")
for col in df.columns:
    print(f"   - {col}: {df[col].dtype}")

# Identify URL and label columns
url_column = None
label_column = None

# Common URL column names
possible_url_cols = ['url', 'URL', 'Url', 'website', 'domain', 'links']
for col in possible_url_cols:
    if col in df.columns:
        url_column = col
        break

# Common label column names  
possible_label_cols = ['label', 'Label', 'result', 'type', 'class', 'category']
for col in possible_label_cols:
    if col in df.columns:
        label_column = col
        break



📋 Column Information:
   - url: object
   - label: object
   - result: int64


In [5]:
if url_column is None:
    raise ValueError("Could not find URL column! Please check your CSV columns")
if label_column is None:
    raise ValueError("Could not find label column! Please check your CSV columns")

print(f"\n🔍 Identified columns:")
print(f"   URL column: '{url_column}'")
print(f"   Label column: '{label_column}'")

# Analyze label distribution
print(f"\n📊 Label Distribution in your data:")
label_counts = df[label_column].value_counts()
print(label_counts)

# Convert labels to binary (0 = benign, 1 = malicious)
if df[label_column].dtype == 'object':
    # Map string labels to numbers
    unique_labels = df[label_column].unique()
    print(f"\n   Unique labels found: {unique_labels}")
    
    # Intelligent mapping based on common patterns
    label_mapping = {}
    for label in unique_labels:
        label_str = str(label).lower()
        if label_str in ['benign', 'good', 'safe', 'legitimate', 'b', '0']:
            label_mapping[label] = 0
        elif label_str in ['malicious', 'phishing', 'malware', 'bad', 'defacement', 'attack', 'm', '1']:
            label_mapping[label] = 1
        else:
            # Ask user for mapping
            print(f"\n⚠️ Unknown label: '{label}'")
            user_input = input(f"   Is '{label}' malicious? (yes/no): ").lower()
            label_mapping[label] = 1 if user_input in ['yes', 'y'] else 0
    
    df['binary_label'] = df[label_column].map(label_mapping)
else:
    # Assume numeric labels, convert to binary if needed
    if df[label_column].nunique() > 2:
        # Multi-class, ask which is malicious
        print(f"\n⚠️ Multiple classes detected: {df[label_column].unique()}")
        df['binary_label'] = (df[label_column] == df[label_column].max()).astype(int)
    else:
        df['binary_label'] = df[label_column].astype(int)

print(f"\n✅ Binary label conversion:")
print(f"   Benign (0): {(df['binary_label']==0).sum()} samples")
print(f"   Malicious (1): {(df['binary_label']==1).sum()} samples")
print(f"   Balance ratio: {(df['binary_label']==1).sum() / len(df):.2%} malicious")



🔍 Identified columns:
   URL column: 'url'
   Label column: 'label'

📊 Label Distribution in your data:
label
benign       316254
malicious    316254
Name: count, dtype: int64

   Unique labels found: ['benign' 'malicious']

✅ Binary label conversion:
   Benign (0): 316254 samples
   Malicious (1): 316254 samples
   Balance ratio: 50.00% malicious


In [6]:
print("\n🔧 STEP 2: Extracting advanced features from URLs...")

class URLFeatureExtractor:
    """Extract comprehensive features from URLs"""
    
    def __init__(self):
        self.suspicious_tlds = {'tk', 'ml', 'ga', 'cf', 'gq', 'top', 'xyz', 'club', 'online', 
                                 'site', 'space', 'tech', 'review', 'trade', 'webcam', 'science'}
        
    def extract_features(self, url):
        """Extract all features from a single URL"""
        if not isinstance(url, str):
            return self._empty_features()
        
        url = str(url).lower().strip()
        
        # Add protocol if missing
        if not url.startswith(('http://', 'https://')):
            url = 'http://' + url
        
        try:
            parsed = urlparse(url)
            hostname = parsed.hostname or ''
            path = parsed.path or ''
            query = parsed.query or ''
            
            features = {}
            
            # === Length features ===
            features['url_len'] = min(len(url), 1000) / 1000
            features['host_len'] = min(len(hostname), 100) / 100
            features['path_len'] = min(len(path), 500) / 500
            features['query_len'] = min(len(query), 200) / 200
            
            # === Character counts ===
            features['dot_count'] = min(url.count('.'), 20) / 20
            features['hyphen_count'] = min(url.count('-'), 10) / 10
            features['underscore_count'] = min(url.count('_'), 10) / 10
            features['slash_count'] = min(url.count('/'), 15) / 15
            features['question_count'] = min(url.count('?'), 5) / 5
            features['equal_count'] = min(url.count('='), 10) / 10
            features['at_count'] = min(url.count('@'), 3) / 3
            
            # === Ratio features ===
            digits = sum(c.isdigit() for c in url)
            letters = sum(c.isalpha() for c in url)
            features['digit_ratio'] = digits / max(len(url), 1)
            features['letter_ratio'] = letters / max(len(url), 1)
            features['special_ratio'] = (len(url) - digits - letters) / max(len(url), 1)
            
            # === Security indicators ===
            features['has_https'] = 1.0 if parsed.scheme == 'https' else 0.0
            features['has_ip'] = 1.0 if re.search(r'\d+\.\d+\.\d+\.\d+', hostname) else 0.0
            features['has_port'] = 1.0 if parsed.port else 0.0
            features['has_at'] = 1.0 if '@' in url else 0.0
            
            # === Suspicious patterns ===
            features['has_suspicious_tld'] = 1.0 if any(tld in hostname for tld in self.suspicious_tlds) else 0.0
            features['has_double_slash'] = 1.0 if '//' in path else 0.0
            features['has_double_hyphen'] = 1.0 if '--' in url else 0.0
            
            # === Subdomain analysis ===
            subdomains = hostname.split('.')
            features['subdomain_count'] = min(len(subdomains) - 1, 5) / 5
            
            # === Path features ===
            features['path_depth'] = min(path.count('/'), 10) / 10
            features['has_extension'] = 1.0 if '.' in path.split('/')[-1] else 0.0
            
            # === Entropy ===
            features['entropy'] = self._calculate_entropy(hostname)
            
            # === Specific phishing indicators ===
            phishing_keywords = ['login', 'verify', 'account', 'secure', 'update', 'confirm', 
                                'signin', 'password', 'banking', 'paypal', 'amazon']
            features['has_phishing_keyword'] = 1.0 if any(kw in url for kw in phishing_keywords) else 0.0
            
            return features
            
        except Exception as e:
            print(f"Error extracting features from {url}: {e}")
            return self._empty_features()
    
    def _calculate_entropy(self, text):
        """Calculate Shannon entropy"""
        if not text:
            return 0.0
        prob = [float(text.count(c)) / len(text) for c in set(text)]
        entropy = -sum(p * np.log2(p) for p in prob)
        return min(entropy / 8, 1.0)
    
    def _empty_features(self):
        """Return empty feature dictionary"""
        return {
            'url_len': 0, 'host_len': 0, 'path_len': 0, 'query_len': 0,
            'dot_count': 0, 'hyphen_count': 0, 'underscore_count': 0,
            'slash_count': 0, 'question_count': 0, 'equal_count': 0,
            'at_count': 0, 'digit_ratio': 0, 'letter_ratio': 0,
            'special_ratio': 0, 'has_https': 0, 'has_ip': 0, 'has_port': 0,
            'has_at': 0, 'has_suspicious_tld': 0, 'has_double_slash': 0,
            'has_double_hyphen': 0, 'subdomain_count': 0, 'path_depth': 0,
            'has_extension': 0, 'entropy': 0, 'has_phishing_keyword': 0
        }
    


🔧 STEP 2: Extracting advanced features from URLs...


In [7]:
class URLFeatureExtractor:
    """Extract comprehensive features from URLs"""
    
    def __init__(self):
        self.suspicious_tlds = {'tk', 'ml', 'ga', 'cf', 'gq', 'top', 'xyz', 'club', 'online', 
                                 'site', 'space', 'tech', 'review', 'trade', 'webcam', 'science'}
        
    def extract_features(self, url):
        """Extract all features from a single URL"""
        if not isinstance(url, str):
            return self._empty_features()
        
        url = str(url).lower().strip()
        
        # Add protocol if missing
        if not url.startswith(('http://', 'https://')):
            url = 'http://' + url
        
        try:
            parsed = urlparse(url)
            hostname = parsed.hostname or ''
            path = parsed.path or ''
            query = parsed.query or ''
            
            features = {}
            
            # === Length features ===
            features['url_len'] = min(len(url), 1000) / 1000
            features['host_len'] = min(len(hostname), 100) / 100
            features['path_len'] = min(len(path), 500) / 500
            features['query_len'] = min(len(query), 200) / 200
            
            # === Character counts ===
            features['dot_count'] = min(url.count('.'), 20) / 20
            features['hyphen_count'] = min(url.count('-'), 10) / 10
            features['underscore_count'] = min(url.count('_'), 10) / 10
            features['slash_count'] = min(url.count('/'), 15) / 15
            features['question_count'] = min(url.count('?'), 5) / 5
            features['equal_count'] = min(url.count('='), 10) / 10
            features['at_count'] = min(url.count('@'), 3) / 3
            
            # === Ratio features ===
            digits = sum(c.isdigit() for c in url)
            letters = sum(c.isalpha() for c in url)
            features['digit_ratio'] = digits / max(len(url), 1)
            features['letter_ratio'] = letters / max(len(url), 1)
            features['special_ratio'] = (len(url) - digits - letters) / max(len(url), 1)
            
            # === Security indicators ===
            features['has_https'] = 1.0 if parsed.scheme == 'https' else 0.0
            features['has_ip'] = 1.0 if re.search(r'\d+\.\d+\.\d+\.\d+', hostname) else 0.0
            features['has_port'] = 1.0 if parsed.port else 0.0
            features['has_at'] = 1.0 if '@' in url else 0.0
            
            # === Suspicious patterns ===
            features['has_suspicious_tld'] = 1.0 if any(tld in hostname for tld in self.suspicious_tlds) else 0.0
            features['has_double_slash'] = 1.0 if '//' in path else 0.0
            features['has_double_hyphen'] = 1.0 if '--' in url else 0.0
            
            # === Subdomain analysis ===
            subdomains = hostname.split('.')
            features['subdomain_count'] = min(len(subdomains) - 1, 5) / 5
            
            # === Path features ===
            features['path_depth'] = min(path.count('/'), 10) / 10
            features['has_extension'] = 1.0 if '.' in path.split('/')[-1] else 0.0
            
            # === Entropy ===
            features['entropy'] = self._calculate_entropy(hostname)
            
            # === Specific phishing indicators ===
            phishing_keywords = ['login', 'verify', 'account', 'secure', 'update', 'confirm', 
                                'signin', 'password', 'banking', 'paypal', 'amazon']
            features['has_phishing_keyword'] = 1.0 if any(kw in url for kw in phishing_keywords) else 0.0
            
            return features
            
        except Exception as e:
            print(f"Error extracting features from {url}: {e}")
            return self._empty_features()
    
    def _calculate_entropy(self, text):
        """Calculate Shannon entropy"""
        if not text:
            return 0.0
        prob = [float(text.count(c)) / len(text) for c in set(text)]
        entropy = -sum(p * np.log2(p) for p in prob)
        return min(entropy / 8, 1.0)
    
    def _empty_features(self):  # <-- THIS METHOD WAS MISSING
        """Return empty feature dictionary with all features set to 0"""
        feature_names = self.get_feature_names()
        return {feature: 0.0 for feature in feature_names}
    
    def get_feature_names(self):
        """Return list of feature names"""
        return [
            'url_len', 'host_len', 'path_len', 'query_len', 'dot_count',
            'hyphen_count', 'underscore_count', 'slash_count', 'question_count',
            'equal_count', 'at_count', 'digit_ratio', 'letter_ratio',
            'special_ratio', 'has_https', 'has_ip', 'has_port', 'has_at',
            'has_suspicious_tld', 'has_double_slash', 'has_double_hyphen',
            'subdomain_count', 'path_depth', 'has_extension', 'entropy',
            'has_phishing_keyword'
        ]

# Extract features from all URLs
feature_extractor = URLFeatureExtractor()
print("   Extracting features (this may take a moment)...")

features_list = []
for idx, url in enumerate(df[url_column]):
    if idx % 1000 == 0 and idx > 0:
        print(f"   Processed {idx}/{len(df)} URLs...")
    features = feature_extractor.extract_features(url)
    features_list.append(features)

X = pd.DataFrame(features_list)
y = df['binary_label'].values

print(f"\n✅ Feature extraction complete!")
print(f"   Feature matrix shape: {X.shape}")
print(f"   Features extracted: {len(feature_extractor.get_feature_names())}")

# ============================================================================
# STEP 3: DATA SPLITTING AND PREPROCESSING
# ============================================================================

print("\n📊 STEP 3: Preparing data for training...")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n   Training set: {len(X_train)} samples")
print(f"   Test set: {len(X_test)} samples")
print(f"   Training - Malicious: {(y_train==1).sum()} ({((y_train==1).sum()/len(y_train))*100:.1f}%)")
print(f"   Test - Malicious: {(y_test==1).sum()} ({((y_test==1).sum()/len(y_test))*100:.1f}%)")

# Scale features
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

   Extracting features (this may take a moment)...
   Processed 1000/632508 URLs...
   Processed 2000/632508 URLs...
   Processed 3000/632508 URLs...
   Processed 4000/632508 URLs...
   Processed 5000/632508 URLs...
   Processed 6000/632508 URLs...
   Processed 7000/632508 URLs...
   Processed 8000/632508 URLs...
   Processed 9000/632508 URLs...
   Processed 10000/632508 URLs...
   Processed 11000/632508 URLs...
   Processed 12000/632508 URLs...
   Processed 13000/632508 URLs...
   Processed 14000/632508 URLs...
   Processed 15000/632508 URLs...
   Processed 16000/632508 URLs...
   Processed 17000/632508 URLs...
   Processed 18000/632508 URLs...
   Processed 19000/632508 URLs...
   Processed 20000/632508 URLs...
   Processed 21000/632508 URLs...
   Processed 22000/632508 URLs...
   Processed 23000/632508 URLs...
   Processed 24000/632508 URLs...
   Processed 25000/632508 URLs...
   Processed 26000/632508 URLs...
   Processed 27000/632508 URLs...
   Processed 28000/632508 URLs...
   Pro

In [8]:
if (y_train == 1).sum() / len(y_train) < 0.4 or (y_train == 1).sum() / len(y_train) > 0.6:
    print("\n⚠️ Class imbalance detected. Applying SMOTE...")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    print(f"   After SMOTE: {(y_train_resampled==1).sum()} malicious, {(y_train_resampled==0).sum()} benign")
else:
    X_train_resampled, y_train_resampled = X_train_scaled, y_train

# Feature selection
print("\n🔍 Selecting best features...")
selector = SelectKBest(f_classif, k=min(15, X.shape[1]))
X_train_selected = selector.fit_transform(X_train_resampled, y_train_resampled)
X_test_selected = selector.transform(X_test_scaled)

selected_features = feature_extractor.get_feature_names()
selected_mask = selector.get_support()
selected_features_list = [selected_features[i] for i in range(len(selected_features)) if selected_mask[i]]
print(f"   Selected {len(selected_features_list)} most important features")


🔍 Selecting best features...
   Selected 15 most important features


In [9]:
print("\n🤖 STEP 4: Training models...")

models = {
    'XGBoost': XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss',
        verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300,
        max_depth=10,
        learning_rate=0.1,
        random_state=42,
        verbose=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        max_depth=7,
        learning_rate=0.1,
        random_state=42
    )
}


🤖 STEP 4: Training models...


In [10]:
results = {}
best_model = None
best_accuracy = 0

for name, model in models.items():
    print(f"\n   Training {name}...")
    
    # Train model
    model.fit(X_train_selected, y_train_resampled)
    
    # Predict
    y_pred = model.predict(X_test_selected)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_train_selected, y_train_resampled, cv=5, scoring='accuracy')
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    print(f"      ✓ Accuracy: {accuracy:.4f}")
    print(f"      ✓ Recall: {recall:.4f}")
    print(f"      ✓ F1-Score: {f1:.4f}")
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model
        best_model_name = name



   Training XGBoost...
      ✓ Accuracy: 0.9921
      ✓ Recall: 0.9865
      ✓ F1-Score: 0.9921

   Training LightGBM...
      ✓ Accuracy: 0.9918
      ✓ Recall: 0.9858
      ✓ F1-Score: 0.9917

   Training Random Forest...
      ✓ Accuracy: 0.9901
      ✓ Recall: 0.9822
      ✓ F1-Score: 0.9900

   Training Gradient Boosting...
      ✓ Accuracy: 0.9916
      ✓ Recall: 0.9856
      ✓ F1-Score: 0.9916


In [11]:
print("\n🎯 STEP 5: Creating ensemble model...")

# Create voting classifier with best models
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', models['XGBoost']),
        ('lgbm', models['LightGBM']),
        ('rf', models['Random Forest'])
    ],
    voting='soft'  # Use probability averaging
)

voting_clf.fit(X_train_selected, y_train_resampled)
y_pred_ensemble = voting_clf.predict(X_test_selected)

ensemble_accuracy = accuracy_score(y_test, y_pred_ensemble)
ensemble_precision = precision_score(y_test, y_pred_ensemble)
ensemble_recall = recall_score(y_test, y_pred_ensemble)
ensemble_f1 = f1_score(y_test, y_pred_ensemble)

print(f"\n   Ensemble Model Performance:")
print(f"   ✓ Accuracy: {ensemble_accuracy:.4f}")
print(f"   ✓ Precision: {ensemble_precision:.4f}")
print(f"   ✓ Recall: {ensemble_recall:.4f}")
print(f"   ✓ F1-Score: {ensemble_f1:.4f}")


🎯 STEP 5: Creating ensemble model...

   Ensemble Model Performance:
   ✓ Accuracy: 0.9917
   ✓ Precision: 0.9982
   ✓ Recall: 0.9853
   ✓ F1-Score: 0.9917


In [12]:
print("\n" + "="*80)
print("📈 STEP 6: FINAL RESULTS")
print("="*80)

# Determine best model
if ensemble_accuracy > best_accuracy:
    final_model = voting_clf
    final_model_name = "Voting Ensemble"
    final_accuracy = ensemble_accuracy
    final_precision = ensemble_precision
    final_recall = ensemble_recall
    final_f1 = ensemble_f1
else:
    final_model = best_model
    final_model_name = best_model_name
    final_accuracy = best_accuracy
    final_precision = results[best_model_name]['precision']
    final_recall = results[best_model_name]['recall']
    final_f1 = results[best_model_name]['f1']

print(f"\n🏆 BEST MODEL: {final_model_name}")
print(f"\n📊 PERFORMANCE METRICS:")
print(f"   ✅ Accuracy:  {final_accuracy:.4f} ({final_accuracy*100:.2f}%)")
print(f"   ✅ Precision: {final_precision:.4f} ({final_precision*100:.2f}%)")
print(f"   ✅ Recall:    {final_recall:.4f} ({final_recall*100:.2f}%)")
print(f"   ✅ F1-Score:  {final_f1:.4f} ({final_f1*100:.2f}%)")

# Confusion Matrix
y_pred_final = final_model.predict(X_test_selected)
cm = confusion_matrix(y_test, y_pred_final)

print(f"\n📊 CONFUSION MATRIX:")
print(f"                  Predicted")
print(f"                 Benign  Malicious")
print(f"   Actual Benign    {cm[0,0]:5d}    {cm[0,1]:5d}")
print(f"          Malicious  {cm[1,0]:5d}    {cm[1,1]:5d}")


📈 STEP 6: FINAL RESULTS

🏆 BEST MODEL: XGBoost

📊 PERFORMANCE METRICS:
   ✅ Accuracy:  0.9921 (99.21%)
   ✅ Precision: 0.9977 (99.77%)
   ✅ Recall:    0.9865 (98.65%)
   ✅ F1-Score:  0.9921 (99.21%)

📊 CONFUSION MATRIX:
                  Predicted
                 Benign  Malicious
   Actual Benign    63107      144
          Malicious    854    62397


In [13]:
false_negatives = cm[1,0]
true_positives = cm[1,1]
false_negative_rate = false_negatives / (false_negatives + true_positives) if (false_negatives + true_positives) > 0 else 0
false_positives = cm[0,1]
true_negatives = cm[0,0]
false_positive_rate = false_positives / (false_positives + true_negatives) if (false_positives + true_negatives) > 0 else 0

print(f"\n🎯 CRITICAL METRICS:")
print(f"   ❌ False Negative Rate (Missed Malicious): {false_negative_rate:.4%}")
print(f"   ⚠️  False Positive Rate (False Alarms): {false_positive_rate:.4%}")

# Check if we achieved 95% accuracy
if final_accuracy >= 0.95:
    print(f"\n🎉 SUCCESS! Achieved {final_accuracy*100:.2f}% accuracy (Target: 95%+)")
else:
    print(f"\n⚠️ Current accuracy: {final_accuracy*100:.2f}%")
    print(f"   Need: 95%+")
    print(f"   Gap: {95 - final_accuracy*100:.2f}%")
    
    # Improvement suggestions
    print(f"\n💡 SUGGESTIONS TO IMPROVE ACCURACY:")
    print(f"   1. Add more training data (current: {len(df)} samples)")
    print(f"   2. Extract additional features (currently {X.shape[1]} features)")
    print(f"   3. Try deep learning with URL embeddings")
    print(f"   4. Use external data sources (WHOIS, DNS, etc.)")

# Detailed classification report
print(f"\n📋 DETAILED CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_final, target_names=['Benign', 'Malicious']))



🎯 CRITICAL METRICS:
   ❌ False Negative Rate (Missed Malicious): 1.3502%
   ⚠️  False Positive Rate (False Alarms): 0.2277%

🎉 SUCCESS! Achieved 99.21% accuracy (Target: 95%+)

📋 DETAILED CLASSIFICATION REPORT:
              precision    recall  f1-score   support

      Benign       0.99      1.00      0.99     63251
   Malicious       1.00      0.99      0.99     63251

    accuracy                           0.99    126502
   macro avg       0.99      0.99      0.99    126502
weighted avg       0.99      0.99      0.99    126502



In [14]:
print("\n💾 STEP 7: Saving model and artifacts...")

# Save all necessary components
model_artifacts = {
    'model': final_model,
    'scaler': scaler,
    'selector': selector,
    'feature_extractor': feature_extractor,
    'selected_features': selected_features_list,
    'training_date': datetime.now(),
    'accuracy': final_accuracy,
    'recall': final_recall,
    'precision': final_precision,
    'f1_score': final_f1,
    'false_negative_rate': false_negative_rate,
    'training_samples': len(df),
    'model_type': final_model_name
}

joblib.dump(model_artifacts, 'url_phishing_detector.pkl')
print("   ✅ Model saved as 'url_phishing_detector.pkl'")

# Save results to CSV
results_df = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': results[name]['accuracy'],
        'Precision': results[name]['precision'],
        'Recall': results[name]['recall'],
        'F1': results[name]['f1'],
        'CV_Mean': results[name]['cv_mean']
    }
    for name in results.keys()
])


💾 STEP 7: Saving model and artifacts...
   ✅ Model saved as 'url_phishing_detector.pkl'


In [15]:
results_df = pd.concat([
    results_df,
    pd.DataFrame([{
        'Model': 'Voting Ensemble',
        'Accuracy': ensemble_accuracy,
        'Precision': ensemble_precision,
        'Recall': ensemble_recall,
        'F1': ensemble_f1,
        'CV_Mean': None
    }])
], ignore_index=True)

results_df.to_csv('model_comparison_results.csv', index=False)
print("   ✅ Results saved as 'model_comparison_results.csv'")

# ============================================================================
# STEP 8: TEST ON SAMPLE URLS
# ============================================================================

print("\n🧪 STEP 8: Testing on sample URLs...")

def predict_url(url, model_artifacts):
    """Predict if a URL is malicious"""
    features = model_artifacts['feature_extractor'].extract_features(url)
    X = pd.DataFrame([features])
    X_scaled = model_artifacts['scaler'].transform(X)
    X_selected = model_artifacts['selector'].transform(X_scaled)
    prediction = model_artifacts['model'].predict(X_selected)[0]
    probability = model_artifacts['model'].predict_proba(X_selected)[0]
    
    return {
        'url': url,
        'prediction': 'Malicious' if prediction == 1 else 'Benign',
        'confidence': max(probability),
        'risk_score': probability[1]
    }


   ✅ Results saved as 'model_comparison_results.csv'

🧪 STEP 8: Testing on sample URLs...


In [16]:
sample_urls = [
    "https://www.google.com",
    "https://github.com",
    "https://www.paypal.com",
    "http://paypal-verify-account.xyz",
    "https://secure-login-page.com/verify",
    "http://192.168.1.100/banking"
]

print("\n   Sample predictions:")
for url in sample_urls:
    result = predict_url(url, model_artifacts)
    status_icon = "🚨" if result['prediction'] == 'Malicious' else "✅"
    print(f"\n   {status_icon} URL: {result['url']}")
    print(f"      Prediction: {result['prediction']}")
    print(f"      Confidence: {result['confidence']:.2%}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("🎯 TRAINING COMPLETE - SUMMARY")
print("="*80)
print(f"\n📊 Dataset: balanced_urls.csv")
print(f"   Total samples: {len(df)}")
print(f"   Benign: {(y==0).sum()} | Malicious: {(y==1).sum()}")
print(f"\n🏆 Best Model: {final_model_name}")
print(f"   Accuracy: {final_accuracy*100:.2f}%")
print(f"   Recall: {final_recall*100:.2f}% (Critical for catching malicious URLs)")
print(f"\n📁 Files saved:")
print(f"   - url_phishing_detector.pkl (Main model)")
print(f"   - model_comparison_results.csv (Performance comparison)")

if final_accuracy >= 0.95:
    print(f"\n🎉✅ MODEL DEPLOYMENT READY! Achieved 95%+ accuracy!")
else:
    print(f"\n⚠️ Current accuracy: {final_accuracy*100:.2f}%")
    print(f"   To reach 95%+, consider:")
    print(f"   1. Add more training data")
    print(f"   2. Review data quality and labels")
    print(f"   3. Enable deep learning features in config")


   Sample predictions:

   ✅ URL: https://www.google.com
      Prediction: Benign
      Confidence: 98.44%

   🚨 URL: https://github.com
      Prediction: Malicious
      Confidence: 99.30%

   ✅ URL: https://www.paypal.com
      Prediction: Benign
      Confidence: 92.80%

   🚨 URL: http://paypal-verify-account.xyz
      Prediction: Malicious
      Confidence: 100.00%

   🚨 URL: https://secure-login-page.com/verify
      Prediction: Malicious
      Confidence: 99.80%

   🚨 URL: http://192.168.1.100/banking
      Prediction: Malicious
      Confidence: 100.00%

🎯 TRAINING COMPLETE - SUMMARY

📊 Dataset: balanced_urls.csv
   Total samples: 632508
   Benign: 316254 | Malicious: 316254

🏆 Best Model: XGBoost
   Accuracy: 99.21%
   Recall: 98.65% (Critical for catching malicious URLs)

📁 Files saved:
   - url_phishing_detector.pkl (Main model)
   - model_comparison_results.csv (Performance comparison)

🎉✅ MODEL DEPLOYMENT READY! Achieved 95%+ accuracy!


In [17]:
# Add GitHub to whitelist or retrain with more GitHub examples

# Option 1: Add GitHub and similar safe domains to training data
additional_benign_urls = [
    'https://github.com',
    'https://gitlab.com',
    'https://bitbucket.org',
    'https://stackoverflow.com',
    'https://medium.com',
    'https://dev.to',
    'https://codesandbox.io',
    'https://replit.com',
    'https://glitch.com',
    'https://heroku.com'
]

# Add to your dataset and retrain
new_benign_df = pd.DataFrame({'url': additional_benign_urls, 'binary_label': 0})
df_updated = pd.concat([df, new_benign_df], ignore_index=True)

# Retrain with expanded dataset
print("Retraining with additional benign URLs...")
# ... run training again

Retraining with additional benign URLs...
